In [6]:
!pip install -e ../. 

Obtaining file:///Users/fabian/Python/REANIMATOR%20SIGIR/Reanimator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for reanimator (pyproject.toml) ... done
  Created wheel for reanimator: filename=reanimator-0.1.4-0.editable-py3-none-any.whl size=5586 sha256=70bfb7224c512161d1170319a97e2541cfaae7bbf591666372fc067fa4637722
  Stored in directory: /private/var/folders/n_/ndw79bp52bx29ynt1q6l44rw0000gn/T/pip-ephem-wheel-cache-jwwdyc9m/wheels/d2/3d/11/4f5c8951501fef9f5bf6fa1404717b91741fc20e3ea1e6b7d7
Successfully built reanimator
  Attempting uninstall: reanimator
    Found existing installation: reanimator 0.1.4
    Uninstalling reanimator-0.1.4:
      Successfully uninstalled reanimator-0.1.4

[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import pyterrier as pt
import nltk

load_dotenv()
nltk.download('punkt_tab')


import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/fabian/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/src/reanimator/sources.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
#filter docs with original rel judgments to decrease candidate pool for faster processing in tutorial
#take only 40 docs and topic 42
only_k = 40
topic_id = "42"

human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()
topic = [t for t in topics if t.query_id == topic_id][0]

doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == topic_id]
len(doc_ids)

In [ ]:
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

#set accelerator options, device could be MPS, CUDA, CPU
accelerator_options = AcceleratorOptions(
        num_threads=8, device=AcceleratorDevice.MPS
    )

In [5]:
docs = reanimator.load_documents(doc_ids=t1_doc_ids)[:10]


Step 1: Loading documents from source...


cord19/trec-covid documents: 100%|██████████| 192509/192509 [00:01<00:00, 172446.58it/s]


In [6]:
reanimator.download_documents(docs)
reanimator.extract_content(docs)
reanimator.save_documents(docs, "documents")


Step 2: Fetching URLs and downloading PDFs...
All DOIs already have cached URLs.


Failed to download 10.1093/nar/gkq1013: 403 Client Error: Forbidden for url: https://academic.oup.com/nar/article-pdf/39/suppl_1/D569/18784692/gkq1013.pdf
Failed to download 10.1093/nar/gkq089: 403 Client Error: Forbidden for url: https://academic.oup.com/nar/article-pdf/38/9/e111/33236399/gkq089.pdf
Failed to download 10.1155/2011/284795: 403 Client Error: Forbidden for url: https://downloads.hindawi.com/journals/ecam/2011/284795.pdf
PDF downloading complete.

Step 3: Extracting content from PDFs...


Extracting Content:   0%|          | 0/10 [00:00<?, ?it/s]/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory'

Saving 10 documents to directory documents...
Finished saving documents.


In [7]:
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

In [8]:
docs = reanimator.load_documents_from_file("documents")

Attempting to load documents from 10 files...
Successfully loaded 10 documents.


In [9]:
labeler = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"), prompt_path="/Users/fabian/Python/REANIMATOR SIGIR/Reanimator/src/reanimator/default_prompt.txt")
model = labeler.model

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [ ]:
#set different chunk modality types

table_chunks = [c for c in chunks if c.modality == "table"]
text_chunks = [c for c in chunks if c.modality == "text"]

print(f"{len(table_chunks)} table chunks")
print(f"{len(text_chunks)} text chunks")

INFO: LocalModelLabeler initialized with model: qwen/qwen3-30b-a3b at http://192.168.178.180:1234/v1


In [10]:
#generate indices for different chunk modalities

indexer_both = Indexer(index_type="bm25", path="data/indices/bm25_both/")
indexer_table = Indexer(index_type="bm25", path="data/indices/bm25_table/")
indexer_text = Indexer(index_type="bm25", path="data/indices/bm25_text/")

indexer_both.index(chunks)
indexer_table.index(table_chunks)
indexer_text.index(text_chunks)

In [11]:
retriever_both = Retriever(indexer=indexer_both)
retriever_table = Retriever(indexer=indexer_table)
retriever_text = Retriever(indexer=indexer_text)

Generating Judgements: 100%|██████████| 1057/1057 [01:27<00:00, 12.10it/s]


In [12]:
from reanimator.models import save_judgements
model = model.replace("/", "_")
save_judgements(machine_judgements, f"machine_{model}_judgements.json")

## Human Relevance Judgments


In [13]:
from reanimator.human_labeling import *

In [14]:
# Load machine labeled pairs here:
with open('machine_gpt-4.1-mini-2025-04-14_judgements.json') as f:
    machine_judgements = json.load(f)

In [15]:
# insert path to your labeling log file 
output_path = "fabian_judgements.json"
# what modality are you labeling: text or table
modality = "text"

In [16]:
label_chunks = [a.to_dict() for a in chunks]
label_topics = [t.to_dict() for t in topics]

In [17]:
to_label = load_label_pairs(machine_judgements, label_chunks, output_path="fabian_judgements.json", modality=modality, num_pairs=20)

17  texts left to label: 20 pairs to label, 3 already labeled in 'fabian_judgements.json'.


In [18]:
labeling_interface(output_path="fabian_judgements.json", sampled=to_label, topics=label_topics, chunks=label_chunks)

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

...

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

In [ ]:
#set batch to for labeling which appear in the pool

batch = [TopicChunkPair(topic=topic, chunk=chunk) for chunk in chunks if chunk.chunk_id in pool]
len(batch)

In [ ]:
#generate synthetic rel judgements

qwen_judgements = await labeler_qwen.label_all(batch)

model_name = labeler_qwen.model.replace("/", "_")
save_judgements(qwen_judgements, f"data/judgments/machine_{model_name}_judgements.json")

In [ ]:
gpt_judgements = await labeler_gpt41mini.label_all(batch)

model_name = labeler_gpt41mini.model.replace("/", "_")
save_judgements(gpt_judgements, f"data/judgments/machine_{model_name}_judgements.json")

In [ ]:
#calculate cohens kappa between two labelers

kappa = calculate_cohens_kappa("data/judgments/machine_gpt-4.1-mini-2025-04-14_judgements.json", "data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
qwen_judgements = load_judgements("data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
run_experiment(rankings=[res_both, res_table, res_text], topics=topics, judgements=qwen_judgements, eval_metrics=[pt.measures.nDCG, pt.measures.P@20], names=["BM25", "BM25 Table", "BM25 Text"])